# Extended Thinking

*Turning reasoning on, calibrating effort, reading it back correctly.*

- Prompting shapes **what** Claude produces. Extended thinking shapes **how much work** it does first.
- On: model writes step-by-step reasoning, then the answer.
- Two jobs — decide when the extra cost is worth it, handle the reasoning it sends back.

<details>
<summary><i>Full module text — intro</i></summary>

### Extended Thinking: Turning reasoning on, calibrating effort, and reading it back correctly

The prompting techniques shape what Claude produces. Extended thinking shapes how much work
Claude does before it answers. Turn it on, and the model writes out its step-by-step
reasoning first, then gives you the final answer. Your job is to decide when that extra work
is worth the cost and to handle the reasoning it sends back.

</details>

## What it does

- Reasoning returns as its own **`thinking` block**, positioned just ahead of the answer block.
- Newest models: thinking content **omitted by default** — request a summary via `display`.
- Reasoning is **adaptive** — you enable it, the model decides how much each request needs.
- Tune depth with **`effort`**, not a fixed token budget.
- `budget_tokens` — deprecated; **400** on newest generations.
- **Thinking tokens cost the same as output tokens.** High effort on a simple task = paying for accuracy you don't need.
- Rule: *match the tool to the task.* Not a default — apply strategically.

<details>
<summary><i>Full module text — What extended thinking does</i></summary>

When you turn on extended thinking, the model "thinks out loud" before it responds. You'll
see this reasoning come back as its own thinking block in the API response, positioned just
ahead of the block that holds the actual answer. On the newest models, the thinking block's
content is omitted by default; you must request a readable summary through the display
setting to see it.

On current models reasoning is adaptive: you enable it with the thinking parameter where it
is not already on by default, and the model decides how much reasoning each request needs.
You tune depth with the effort setting rather than a fixed token budget. The older
budget_tokens control is deprecated and, on the newest model generations, returns a 400
error.

That reasoning isn't free; thinking tokens cost the same as output tokens, so running a
simple task at high effort means paying for accuracy you don't need. The choice here mirrors
the one you have already made: match the tool to the task. Don't reach for extended thinking
by default, apply it strategically where needed.

</details>

In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()

# NOTE: Opus 5 demonstrates every point in this lesson — thinking on by default, the full
# low -> max effort range, and the budget_tokens 400. The rest of this course uses
# claude-sonnet-5, which behaves the same for all of it; swap the string to run cheaper.
model = "claude-opus-5"

### Response = list of blocks

`response.content` is a list, not a string. `thinking` first, `text` second.
Branch on `block.type` — `content[0].text` breaks the moment reasoning is on.

In [3]:
PUZZLE = (
    "A shop sells apples in bags of 6 and 11. "
    "What is the largest number of apples you cannot buy exactly? "
    "Give the number and one sentence of justification."
)

response = client.messages.create(
    model=model,
    max_tokens=4000,
    thinking={"type": "adaptive", "display": "summarized"},
    messages=[{"role": "user", "content": PUZZLE}],
)

# The block order is the lesson here: thinking first, answer second.
[block.type for block in response.content]

['thinking', 'text']

In [4]:
for block in response.content:
    if block.type == "thinking":
        print("--- THINKING (summary) ---")
        print(block.thinking)
    elif block.type == "text":
        print("--- ANSWER ---")
        print(block.text)

--- THINKING (summary) ---
Using the Chicken McNugget theorem for coprime numbers 6 and 11, the largest non-representable number is 6·11 − 6 − 11 = 49. I verify this by checking residues mod 6 across multiples of 11, confirming none of them can express 49 using nonnegative combinations of 6 and 11, while every number 50 and above can be represented.


--- ANSWER ---
## Answer: 49

**Justification:** Since 6 and 11 are coprime, the Frobenius (Chicken McNugget) number is $6 \times 11 - 6 - 11 = 49$; concretely, 49 fails because subtracting 0, 11, 22, 33, or 44 leaves 49, 38, 27, 16, or 5 — none a multiple of 6 — while every number from 50 upward is buyable (50 = 6·4 + 11·2, 51 = 6·1 + 11·... actually 51 = 6·... ) let me state it cleanly: the smallest buyable amounts in each residue class mod 6 are 0, 55, 44, 33, 22, 11 (for residues 0,1,2,3,4,5), and the largest of these is 55, so every number ≥ 50 is buyable and 55 − 6 = 49 is the largest that is not.


### `display` — visibility only

| `display` | Block | Text inside |
|---|---|---|
| `"omitted"` (default) | present | `""` |
| `"summarized"` | present | readable summary |

- Raw chain of thought is **never** returned. Summary ≠ transcript.
- Thinking happens, and is **billed**, the same either way.

In [5]:
omitted = client.messages.create(
    model=model,
    max_tokens=4000,
    thinking={"type": "adaptive"},  # display defaults to "omitted"
    messages=[{"role": "user", "content": PUZZLE}],
)

thinking_block = next(b for b in omitted.content if b.type == "thinking")

print(f"block present:  {thinking_block.type}")
print(f"thinking text:  {thinking_block.thinking!r}")
print(f"output tokens:  {omitted.usage.output_tokens}  <- thinking counted here regardless")

block present:  thinking
thinking text:  ''
output tokens:  545  <- thinking counted here regardless


Streaming to users? The default reads as a long pause before any output. Set `"summarized"` explicitly.

## Calibrating effort

`output_config.effort` — `low` | `medium` | `high` | `xhigh` | `max`. Defaults to `high`.

Replaced the fixed budget: **you set the ambition, the model sets the amount.**

In [6]:
import time

def measure(effort, prompt):
    """Run one request at a given effort and report what it cost."""
    started = time.time()
    response = client.messages.create(
        model=model,
        max_tokens=8000,
        thinking={"type": "adaptive"},
        output_config={"effort": effort},
        messages=[{"role": "user", "content": prompt}],
    )
    answer = next((b.text for b in response.content if b.type == "text"), "")
    return {
        "effort": effort,
        "output_tokens": response.usage.output_tokens,
        "seconds": round(time.time() - started, 1),
        "answer": answer.strip().replace("\n", " ")[:80],
    }

In [7]:
# A hard prompt, where more reasoning plausibly buys something.
for r in [measure(e, PUZZLE) for e in ["low", "high", "max"]]:
    print(f"{r['effort']:>5}  {r['output_tokens']:>6} out tok  {r['seconds']:>5}s  {r['answer']}")

  low     236 out tok    5.0s  **49 apples.**  Since 6 and 11 are coprime, the Chicken McNugget (Frobenius) for
 high     532 out tok    7.5s  ## Answer: 49  **Justification:** Since 6 and 11 are coprime, the Chicken McNugg
  max     498 out tok    7.3s  **49.**  Since 6 and 11 are coprime, the Frobenius (Chicken McNugget) formula gi


- Output tokens climb steeply with effort — **thinking tokens are output tokens**.
- At this problem size the answer is usually identical at every level.

→ Don't default to `max`. Same sweep on a lookup task, where the reasoning is waste:

In [8]:
EASY = "What is the capital of Australia? Answer with just the city name."

for r in [measure(e, EASY) for e in ["low", "max"]]:
    print(f"{r['effort']:>5}  {r['output_tokens']:>6} out tok  {r['seconds']:>5}s  {r['answer']}")

  low       6 out tok    2.0s  Canberra
  max      35 out tok    1.5s  Canberra


### Cost

- Billed at the **output** rate — no discounted reasoning tier.
- Opus 5: **$5 / MTok in, $25 / MTok out**.
- Cost tracks how much the model chose to think → controlled by `effort`.

In [9]:
INPUT_PER_MTOK = 5.00   # Opus 5
OUTPUT_PER_MTOK = 25.00

def cost(usage):
    return (usage.input_tokens * INPUT_PER_MTOK
            + usage.output_tokens * OUTPUT_PER_MTOK) / 1_000_000

for effort in ["low", "max"]:
    r = client.messages.create(
        model=model,
        max_tokens=8000,
        thinking={"type": "adaptive"},
        output_config={"effort": effort},
        messages=[{"role": "user", "content": PUZZLE}],
    )
    print(f"{effort:>5}  in={r.usage.input_tokens:>4}  out={r.usage.output_tokens:>6}  ${cost(r.usage):.4f}")

  low  in=  56  out=   238  $0.0062
  max  in=  56  out=   549  $0.0140


### `budget_tokens` is deprecated

Old: `thinking={"type": "enabled", "budget_tokens": N}` — a hard ceiling.
Newest models **reject it (400)** rather than ignore it, so old code fails loudly.

In [10]:
import anthropic

try:
    client.messages.create(
        model=model,
        max_tokens=4000,
        thinking={"type": "enabled", "budget_tokens": 2000},  # the old API
        messages=[{"role": "user", "content": PUZZLE}],
    )
except anthropic.BadRequestError as e:
    print(f"HTTP {e.status_code}")
    print(e.message)

HTTP 400
Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '"thinking.type.enabled" is not supported for this model. Use "thinking.type.adaptive" and "output_config.effort" to control thinking behavior.'}, 'request_id': 'req_011Cen8GC3KruQJhNTU5inEB'}


**Migrating:** drop `budget_tokens` → `"adaptive"` + an `effort` level.
Small budget ≈ `low`/`medium`; large ≈ `high`/`xhigh`. No exact mapping — that's the point.

```python
# before
thinking={"type": "enabled", "budget_tokens": 10000}
# after
thinking={"type": "adaptive"}, output_config={"effort": "high"}
```

## When to use

| Task shape | Call | Why |
|---|---|---|
| Multi-step reasoning — math derivation, multi-hop logic, planning dependent actions | **Enable**, effort matched to depth | The reasoning pass works through dependencies it would otherwise skip |
| Mechanical / lookup — classification, format conversion, field extraction, short facts | **Leave off** | No better answer, more tokens. A constrained prompt is the right tool |
| Agentic loops planning across tool calls | **Enable**, budget the planning step not each call | Fewer wrong-tool selections downstream. Carry-back rule applies |

<details>
<summary><i>Full module text — When to use extended thinking</i></summary>

| Task shape | Extended thinking call | Reason |
|---|---|---|
| Multi-step reasoning where the model has to hold several constraints at once: a math derivation, a multi-hop logic problem, planning a sequence of dependent actions. | Enable it, with the effort level matched to the depth of the problem. | The reasoning pass is where the model works through dependencies it would otherwise skip. |
| Mechanical or lookup tasks: classification, format conversion, extracting a field, short factual answers. | Leave it off. | Extended thinking will not improve the answer, and you will be paying more tokens for something you didn't need. A bare prompt with an output constraint is the right tool. |
| Agentic loops where the model plans across several tool calls. | Enable it and budget for the planning step rather than per call. | Reasoning before a plan reduces wrong-tool selection downstream. Note the carry-back rule below, which applies in every tool-use loop. |

</details>

## ⚠️ Carry-back rule

**Every thinking block goes back to the API exactly as it arrived.**

- Applies whenever thinking **and** tools are both on.
- Each block carries a **`signature`** confirming the reasoning wasn't tampered with.
- Edit it, summarize it, or drop it → signature stops matching → request rejected.
- **Redacted** thinking blocks: encrypted, not for human reading, still returned untouched.
- Structural requirement, not a prompting choice.

<details>
<summary><i>Full module text — The carry-back rule</i></summary>

**The carry-back rule: thinking blocks must return to the API unchanged**

When extended thinking is on and your conversation uses tools, there's one rule you can't
skip: every thinking block you get back has to go back to the API exactly as it arrived on
the next turn. Each block comes with a signature that confirms the reasoning wasn't tampered
with. If you edit it, summarize it, or drop it, the signature stops matching and the API
rejects the request.

Redacted thinking blocks work the same way. Their contents are encrypted and not meant to be
read by humans, but they still have to be returned untouched.

This is a structural requirement, not a prompting choice you get to make. The most common
slip-up is stripping out the thinking block to save context, which ends up breaking your
next request. If the real worry is how much context piles up from accumulated reasoning, the
fix is the context-engineering work we'll cover in this module.

</details>

In [11]:
# Every thinking block ships with a signature. A redacted one carries `data` instead of
# readable `thinking` text — either way, it goes back untouched.
tb = next(b for b in response.content if b.type == "thinking")

print("block type:", tb.type)
print("fields:    ", [f for f in ("thinking", "signature", "data") if hasattr(tb, f)])
print("signature: ", tb.signature[:40], "...")

block type: thinking
fields:     ['thinking', 'signature']
signature:  CAIS7QUKjgEIERgCKkAurD3HkM1V/WxamMeQihXD ...


In practice: append **`response.content` whole**. Never rebuild the assistant turn from just the text you cared about.

In [12]:
WEATHER_TOOL = {
    "name": "get_weather",
    "description": "Get the current temperature in Celsius for a city.",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
}

messages = [{
    "role": "user",
    "content": "What's the temperature in Sydney and Melbourne? Which is warmer?",
}]

first = client.messages.create(
    model=model,
    max_tokens=4000,
    thinking={"type": "adaptive", "display": "summarized"},
    tools=[WEATHER_TOOL],
    messages=messages,
)

print("stop_reason:", first.stop_reason)
print("blocks:     ", [b.type for b in first.content])

stop_reason: tool_use
blocks:      ['text', 'tool_use', 'tool_use']


In [13]:
READINGS = {"Sydney": 22.5, "Melbourne": 18.0}

# THE CARRY-BACK: append the assistant turn WHOLE — thinking blocks included, untouched.
messages.append({"role": "assistant", "content": first.content})

# All tool_results for one assistant turn go back in a SINGLE user message.
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": str(READINGS.get(block.input["city"], "unknown")),
        }
        for block in first.content if block.type == "tool_use"
    ],
})

second = client.messages.create(
    model=model,
    max_tokens=4000,
    thinking={"type": "adaptive", "display": "summarized"},
    tools=[WEATHER_TOOL],
    messages=messages,
)

print(next(b.text for b in second.content if b.type == "text"))

Here are the current temperatures:

- **Sydney:** 22.5 °C
- **Melbourne:** 18.0 °C

**Sydney is warmer**, by 4.5 °C.


### Stripping them breaks the next request

The common slip-up: dropping thinking blocks to save context. Same conversation as above, thinking removed:

In [14]:
# Same history, thinking blocks removed from the assistant turn.
stripped = [b for b in first.content if b.type != "thinking"]
broken = [messages[0], {"role": "assistant", "content": stripped}, messages[2]]

try:
    client.messages.create(
        model=model,
        max_tokens=4000,
        thinking={"type": "adaptive", "display": "summarized"},
        tools=[WEATHER_TOOL],
        messages=broken,
    )
    print("No error raised — inspect the response before assuming this is safe.")
except anthropic.BadRequestError as e:
    print(f"HTTP {e.status_code}")
    print(e.message)

No error raised — inspect the response before assuming this is safe.


Context piling up from accumulated reasoning? Fix it with **context engineering** (later in this module) — not by deleting blocks.

## Not covered here

Model selection. *Whether* to enable reasoning (this lesson) ≠ *which model* to run — see **MSO Foundations**, the module before this one.

<details>
<summary><i>Full module text — Forward pointer</i></summary>

This lesson enables reasoning and calibrates its effort setting; it does not cover model
selection. Choosing which model to run, as distinct from whether to enable reasoning, is
taught in the MSO Foundations module that precedes this one.

</details>

## Summary

| | |
|---|---|
| **Handles well** | Hard reasoning and planning, where a wrong answer is expensive and the extra tokens buy accuracy |
| **Adds cost / complexity** | Carry-back requirement in tool loops; an effort setting to calibrate |
| **Use something else** | Classification, extraction, format tasks — a constrained prompt is cheaper and just as accurate |

<details>
<summary><i>Full module text — Summary</i></summary>

**Handles well**

Hard reasoning and planning tasks where a wrong answer is expensive and the extra tokens buy
accuracy.

**Adds cost or complexity**

The carry-back requirement in tool-use loops, and an effort setting you now must calibrate.

**Use a different approach**

For classification, extraction, and format tasks, a well-constrained prompt is cheaper and
just as accurate.

</details>